# Solar Generation Forecast — Training Notebook

Trains a **multi-step direct XGBoost** model for solar station generation and registers
it to the MLflow Registry, ready to be served by **ml-server** (`model_type: solar`).

**Pipeline**
1. Pull historical generation data from the SCADA API
2. Pull historical weather data from the internal weather service (irradiance + temp + cloud cover)
3. Exploratory data analysis — generation profile, irradiance correlation
4. Feature engineering — lag features + weather features + cyclic time encodings
5. Time-series cross-validation + hyperparameter search
6. Final training — one XGBoost booster per forecast horizon step
7. Evaluation — MAE / RMSE / MAPE per step and averaged
8. Bundle assembly (`bundle/model/` + `bundle/configuration/cache_config.json`)
9. MLflow experiment logging + model registration + `Production` alias
10. Smoke-test via `SolarAdapter`

**Feature convention** (must match `SolarAdapter._build_feature_row`):
```
X_h[i] = [y[i-N_LAGS], ..., y[i-1],          # N_LAGS lag values
           GHI(t+h),                           # solar_radiation  W/m²
           temp(t+h),                          # temperature  °C
           cloud(t+h),                         # cloud_cover  %
           sin(2π·hour/24), cos(2π·hour/24),   # hour-of-day
           sin(2π·doy/365), cos(2π·doy/365)]   # day-of-year
```
Total features per sample = N_LAGS + 7  
Each horizon step h uses **different weather/time features** (matching inference exactly).

**Bundle layout** (matches `ModelProvider._is_bundle_valid`):
```
bundle/
  configuration/cache_config.json   ← runtime config (model_type=solar + weather source)
  model/
    xgb_model.json                  ← manifest {"steps": ["step_0.ubj", ...]}
    step_0.ubj                      ← booster for horizon 0
    step_1.ubj
    ...
```

## 0 · Imports

In [1]:
import json
import math
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient
import numpy as np
import pandas as pd
import requests
import xgboost as xgb
from sklearn.model_selection import ParameterSampler

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True

print(f'xgboost {xgb.__version__}  |  mlflow {mlflow.__version__}')

xgboost 3.2.0  |  mlflow 3.11.1


## 1 · Configuration

**Edit this cell before every training run.**

In [ ]:
# ── SCADA / NDC ──────────────────────────────────────────────────────────────
SCADA_URL = "http://localhost:7080/api/v1/read/archives"

# Archive tag for solar generation (MW or kWh)
ARCHIVES = ["/root/FP/PROJECT/AKMOLA/Nura_SES/Pgen_sum/archives/out_value"]

# How many days of history to pull for training
TRAIN_HISTORY_DAYS = 180    # solar needs at least 90 days (ideally 365+)

# ── Weather service ───────────────────────────────────────────────────────────
# Internal weather service base URL (same as WEATHER_API_URL env var in ml-server)
WEATHER_BASE_URL = "http://localhost:8050/api/v1/weather"

# Coordinates of the solar station
STATION_LAT = 51.18
STATION_LON = 71.45

# Weather variables to fetch and use as features (order matters — must match SolarAdapter)
WEATHER_VARIABLES = ["solar_radiation", "temperature", "cloud_cover"]

# ── Time resolution ───────────────────────────────────────────────────────────
STEP_SECONDS = 3600          # seconds per timestep (3600 = hourly)
INPUT_RANGE  = 168           # look-back steps at inference
OUTPUT_RANGE = 24            # forecast horizon (one booster per step)

# ── Feature engineering ───────────────────────────────────────────────────────
# Lag window for training; should be <= INPUT_RANGE.
# Total features = N_LAGS + 7  (3 weather + 4 time).
# Stored in booster and used by SolarAdapter at inference — do not change
# after training without retraining.
N_LAGS = 168
N_WEATHER_FEATURES = 3   # must match SolarAdapter.N_WEATHER_FEATURES
N_TIME_FEATURES    = 4   # must match SolarAdapter.N_TIME_FEATURES
N_FIXED = N_WEATHER_FEATURES + N_TIME_FEATURES   # = 7
N_FEATURES = N_LAGS + N_FIXED

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://localhost:5050"      # or "sqlite:///mlflow.db"
EXPERIMENT_NAME     = "solar/STATION_NURA/generation" # one experiment per station
REGISTERED_MODEL    = "solar_STATION_NURA_generation"  # MLflow registered-model name
REGISTER_ALIAS      = "Production"                 # alias assigned after training

# ── Fallback ───────────────────────────────────────────────────────────────────
FALLBACK_MODEL = "naive"   # "none" | "naive" | "ar"

# ── Cross-validation ──────────────────────────────────────────────────────────
N_CV_SPLITS   = 5
N_TUNE_ITER   = 25
TUNE_ON_STEP  = 0    # horizon step used for HP tuning (0 = next hour)

# ── Output directory ──────────────────────────────────────────────────────────
BUNDLE_DIR = Path("bundle_solar_output")

print('Configuration loaded')
print(f'  station       : lat={STATION_LAT}  lon={STATION_LON}')
print(f'  archives      : {ARCHIVES}')
print(f'  step          : {STEP_SECONDS}s  input={INPUT_RANGE}  output={OUTPUT_RANGE}')
print(f'  n_lags        : {N_LAGS}  →  total features = {N_FEATURES}')
print(f'  MLflow        : {MLFLOW_TRACKING_URI}  experiment={EXPERIMENT_NAME}')

Configuration loaded
  station       : lat=51.18  lon=71.45
  archives      : ['/SOLAR/STATION_1/P_WATT']
  step          : 3600s  input=168  output=24
  n_lags        : 168  →  total features = 175
  MLflow        : http://localhost:5000  experiment=solar/STATION_1/generation


## 2 · Data Loading

Fetches both generation (SCADA) and historical weather for the same period.
Both fall back to synthetic stubs when the services are unavailable.

In [5]:
# ── 2a: SCADA / generation data ───────────────────────────────────────────────

def fetch_scada(
    url: str,
    archives: list[str],
    step_seconds: int,
    days: int,
    request_overrides: dict | None = None,
) -> dict[str, list]:
    """POST to the SCADA API and return the raw payload."""
    now     = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    to_ms   = int(now.timestamp() * 1000)
    from_ms = int((now - timedelta(days=days)).timestamp() * 1000)
    body    = {"from": from_ms, "to": to_ms, "archive": archives, "step": step_seconds}
    if request_overrides:
        body.update(request_overrides)
    resp = requests.post(url, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()


def make_stub_generation(
    archives: list[str],
    step_seconds: int,
    days: int,
    capacity_mw: float = 50.0,
    seed: int = 42,
) -> dict[str, list]:
    """Synthetic solar generation — realistic diurnal + seasonal profile."""
    rng     = np.random.default_rng(seed)
    now     = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    to_ms   = int(now.timestamp() * 1000)
    n       = int(days * 86400 / step_seconds)
    step_ms = step_seconds * 1000
    ts_arr  = np.arange(to_ms - n * step_ms, to_ms, step_ms)

    payload: dict = {}
    for name in archives:
        gen = []
        for t in ts_arr:
            dt   = datetime.fromtimestamp(t / 1000, tz=timezone.utc)
            hour = dt.hour + dt.minute / 60.0
            doy  = dt.timetuple().tm_yday
            # Diurnal: peak at solar noon
            solar_elev  = max(0.0, math.sin(math.pi * (hour - 6) / 12))
            # Seasonal: stronger in summer (Northern hemisphere)
            seasonal    = 0.75 + 0.25 * math.cos(2 * math.pi * (doy - 172) / 365)
            cloud_noise = rng.uniform(0.6, 1.0)
            val = capacity_mw * solar_elev * seasonal * cloud_noise
            gen.append([int(t), round(max(val, 0.0), 3)])
        payload[name] = gen
    return payload


USE_STUB_SCADA = False

try:
    if USE_STUB_SCADA:
        raise RuntimeError('stub mode requested')
    raw_generation = fetch_scada(SCADA_URL, ARCHIVES, STEP_SECONDS, TRAIN_HISTORY_DAYS)
    print(f'SCADA OK — {len(raw_generation)} archive(s)')
except Exception as e:
    print(f'SCADA unavailable ({e}); using synthetic generation stub')
    raw_generation = make_stub_generation(ARCHIVES, STEP_SECONDS, TRAIN_HISTORY_DAYS)

# Parse into DataFrame
frames = []
for arch, series in raw_generation.items():
    ts  = [pt[0] for pt in series]
    val = [pt[1] for pt in series]
    df  = pd.DataFrame({'ts_ms': ts, arch: val})
    frames.append(df.set_index('ts_ms'))

gen_data = frames[0] if len(frames) == 1 else frames[0].join(frames[1:], how='outer')
gen_data.index = pd.to_datetime(gen_data.index, unit='ms', utc=True)
gen_data = gen_data.sort_index()

TARGET_COL = gen_data.columns[0]
series_raw = gen_data[TARGET_COL].copy()

print(f'\nTarget  : {TARGET_COL}')
print(f'Points  : {len(series_raw)}  ({series_raw.index[0]}  →  {series_raw.index[-1]})')
print(f'Missing : {series_raw.isna().sum()} ({series_raw.isna().mean():.1%})')
print(series_raw.describe().to_string())

SCADA OK — 0 archive(s)


IndexError: list index out of range

In [ ]:
# ── 2b: Historical weather data ───────────────────────────────────────────────
#
# The weather service is called for each day-sized chunk because the /historical
# endpoint (if supported) or /forecast endpoint has a request horizon limit.
# Adjust the fetch strategy to match your service's actual API.

def fetch_weather_historical(
    base_url: str,
    lat: float,
    lon: float,
    from_ms: int,
    to_ms: int,
    step_seconds: int = 3600,
    variables: list[str] | None = None,
    timeout: int = 30,
) -> list[dict]:
    """
    Fetch historical hourly weather from the internal weather service.

    Expected response format (same as /forecast):
        {"hourly": [{"timestamp_ms": int, "solar_radiation": float,
                     "temperature": float, "cloud_cover": float}, ...]}

    Falls back to /forecast?... for services that expose only a forecast endpoint.
    """
    params: dict = {
        "lat"  : lat,
        "lon"  : lon,
        "from" : from_ms,
        "to"   : to_ms,
        "step" : step_seconds,
    }
    if variables:
        params["variables"] = ",".join(variables)

    resp = requests.get(f"{base_url}/historical", params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    return data.get("hourly", [])


def make_stub_weather(
    from_ms: int,
    to_ms: int,
    step_seconds: int,
    lat: float,
    seed: int = 99,
) -> list[dict]:
    """
    Physics-based synthetic weather stub.

    Computes theoretical clear-sky GHI from solar geometry,
    then adds random cloud cover attenuation.
    Returns the same format as fetch_weather_historical.
    """
    rng     = np.random.default_rng(seed)
    step_ms = step_seconds * 1000
    ts_arr  = np.arange(from_ms, to_ms, step_ms)
    SOLAR_CONST = 1361.0   # W/m²

    records = []
    for t in ts_arr:
        dt    = datetime.fromtimestamp(t / 1000, tz=timezone.utc)
        hour  = dt.hour + dt.minute / 60.0
        doy   = dt.timetuple().tm_yday

        # Solar declination (degrees)
        decl_rad = math.radians(23.45 * math.sin(math.radians(360 / 365 * (doy - 81))))
        # Hour angle (noon = 0)
        ha_rad   = math.radians((hour - 12) * 15)
        lat_rad  = math.radians(lat)
        sin_elev = (math.sin(lat_rad) * math.sin(decl_rad)
                    + math.cos(lat_rad) * math.cos(decl_rad) * math.cos(ha_rad))
        solar_elev = max(0.0, sin_elev)

        # Clear-sky GHI
        clear_sky_ghi = SOLAR_CONST * solar_elev

        # Random cloud cover attenuation
        cloud_cover = float(rng.uniform(5, 80))   # percent
        cloud_att  = 1.0 - 0.75 * (cloud_cover / 100.0) ** 3.4  # Kasten formula approx
        ghi        = clear_sky_ghi * cloud_att

        # Temperature: diurnal + seasonal
        seasonal_temp = 15 + 15 * math.sin(math.radians(360 / 365 * (doy - 80)))
        diurnal_temp  = 5 * math.sin(math.pi * (hour - 6) / 12)
        temperature   = seasonal_temp + diurnal_temp + rng.normal(0, 1.0)

        records.append({
            "timestamp_ms"       : int(t),
            "solar_radiation": round(max(ghi, 0.0), 2),
            "temperature"     : round(float(temperature), 2),
            "cloud_cover"         : round(float(np.clip(cloud_cover, 0, 100)), 1),
        })
    return records


USE_STUB_WEATHER = False

now     = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
to_ms   = int(now.timestamp() * 1000)
from_ms = int((now - timedelta(days=TRAIN_HISTORY_DAYS)).timestamp() * 1000)

try:
    if USE_STUB_WEATHER:
        raise RuntimeError('stub mode requested')
    hourly_records = fetch_weather_historical(
        WEATHER_BASE_URL, STATION_LAT, STATION_LON,
        from_ms, to_ms, STEP_SECONDS, WEATHER_VARIABLES,
    )
    print(f'Weather service OK — {len(hourly_records)} hourly records')
except Exception as e:
    print(f'Weather service unavailable ({e}); using synthetic stub')
    hourly_records = make_stub_weather(from_ms, to_ms, STEP_SECONDS, STATION_LAT)

# Parse into DataFrame indexed by timestamp_ms
weather_df = pd.DataFrame(hourly_records)

# Normalize: accept both 'timestamp_ms' and 'time' columns
if 'timestamp_ms' in weather_df.columns:
    weather_df = weather_df.set_index('timestamp_ms')
elif 'time' in weather_df.columns:
    weather_df['timestamp_ms'] = pd.to_datetime(weather_df['time']).astype(np.int64) // 10**6
    weather_df = weather_df.set_index('timestamp_ms')

# Ensure required columns exist
for col in WEATHER_VARIABLES:
    if col not in weather_df.columns:
        print(f'  WARNING: weather column "{col}" missing — filling with 0.0')
        weather_df[col] = 0.0

weather_df = weather_df[WEATHER_VARIABLES].sort_index()

# Build a dict for O(1) lookup: {timestamp_ms: np.array([ghi, temp, cloud])}
weather_lookup: dict[int, np.ndarray] = {
    int(ts): weather_df.loc[ts, WEATHER_VARIABLES].values.astype(np.float32)
    for ts in weather_df.index
}

print(f'\nWeather records : {len(weather_df)}')
print(weather_df.describe().to_string())

## 3 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

# Full generation series
axes[0].plot(series_raw.index, series_raw.values, lw=0.5, color='goldenrod')
axes[0].set_title(f'Solar generation history — {TARGET_COL}')
axes[0].set_ylabel('MW')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Last 7 days with GHI overlay
last7_gen = series_raw.last('7D')
ax1 = axes[1]
ax1.plot(last7_gen.index, last7_gen.values, lw=0.9, color='goldenrod', label='Generation')
ax1.set_ylabel('MW')
ax1_r = ax1.twinx()
# Align weather to last-7-days range
w_idx = weather_df.index[(weather_df.index >= last7_gen.index.astype(np.int64).min() // 10**6)
                         & (weather_df.index <= last7_gen.index.astype(np.int64).max() // 10**6)]
if len(w_idx) > 0:
    wt = pd.to_datetime(w_idx, unit='ms', utc=True)
    ax1_r.fill_between(wt, weather_df.loc[w_idx, 'solar_radiation'],
                        alpha=0.25, color='orange', label='GHI')
    ax1_r.set_ylabel('GHI (W/m²)', color='orange')
ax1.set_title('Last 7 days — generation vs irradiance')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b %Hh'))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

# Hourly average profile
hourly_profile = series_raw.groupby(series_raw.index.hour).mean()
axes[2].bar(hourly_profile.index, hourly_profile.values, width=0.8, color='goldenrod')
axes[2].set_title('Average hourly generation profile')
axes[2].set_xlabel('Hour of day (UTC)')
axes[2].set_ylabel('MW')
axes[2].set_xticks(range(24))

for ax in axes:
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# ── Irradiance ↔ generation scatter ──────────────────────────────────────────
# Align generation and GHI by timestamp
gen_ts_ms = (series_raw.index.astype(np.int64) // 10**6).values

aligned_ghi = np.array([
    weather_lookup.get(int(t), np.zeros(3))[0]    # index 0 = solar_radiation
    for t in gen_ts_ms
])

# Only daytime points (GHI > 10 W/m²)
daytime_mask = (aligned_ghi > 10) & series_raw.notna().values
r = np.corrcoef(aligned_ghi[daytime_mask], series_raw.values[daytime_mask])[0, 1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(aligned_ghi[daytime_mask], series_raw.values[daytime_mask],
           s=4, alpha=0.25, color='goldenrod')
ax.set_xlabel('GHI — solar_radiation (W/m²)')
ax.set_ylabel('Solar generation (MW)')
ax.set_title(f'GHI vs generation correlation (daytime only)  r={r:.3f}')
plt.tight_layout()
plt.show()

print(f'GHI-generation Pearson r (daytime) : {r:.4f}')
print(f'Missing generation                 : {series_raw.isna().sum()} / {len(series_raw)}')

## 4 · Feature Engineering

Feature layout **must** match `SolarAdapter._build_feature_row`:

```python
X_h[i] = [y[i-N_LAGS], ..., y[i-1],          # lag features
           GHI(t_i + h),                       # solar_radiation at forecast time
           temp(t_i + h),                      # temperature
           cloud(t_i + h),                     # cloud_cover
           sin(2π·hour/24), cos(2π·hour/24),   # hour-of-day
           sin(2π·doy/365), cos(2π·doy/365)]   # day-of-year
```

Each booster `h` is trained on its own feature matrix `X_h` because the
weather and time features differ per horizon step.

In [ ]:
def interpolate_series(values: np.ndarray) -> np.ndarray:
    """Linear interpolation of NaN gaps (same as HistoricalDataClient)."""
    s = pd.Series(values, dtype=float)
    return s.interpolate(method='linear', limit_direction='both').values


def get_weather_vec(
    lookup: dict[int, np.ndarray],
    ts_ms: int,
    step_ms: int,
    n_vars: int,
) -> np.ndarray:
    """
    Look up weather features for a given timestamp.

    Falls back to the nearest rounded hour if an exact match is missing.
    Returns zeros if no nearby entry exists.
    """
    ts_rounded = int(round(ts_ms / step_ms) * step_ms)
    vec = lookup.get(ts_rounded)
    if vec is None:
        vec = lookup.get(int(ts_ms))
    return vec if vec is not None else np.zeros(n_vars, dtype=np.float32)


def cyclic_time_features(ts_ms: int) -> np.ndarray:
    """
    Cyclic time features — must match SolarAdapter._cyclic_time_features.

    Returns [sin_hour, cos_hour, sin_doy, cos_doy].
    """
    dt  = datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc)
    h   = float(dt.hour) + float(dt.minute) / 60.0
    doy = float(dt.timetuple().tm_yday)
    return np.array(
        [
            math.sin(2 * math.pi * h   / 24.0),
            math.cos(2 * math.pi * h   / 24.0),
            math.sin(2 * math.pi * doy / 365.0),
            math.cos(2 * math.pi * doy / 365.0),
        ],
        dtype=np.float32,
    )


def build_solar_dataset(
    values      : np.ndarray,
    timestamps_ms: np.ndarray,
    weather_lkp : dict[int, np.ndarray],
    n_lags      : int,
    output_range: int,
    step_ms     : int,
    n_weather   : int = 3,
) -> tuple[dict[int, np.ndarray], np.ndarray]:
    """
    Build per-step feature matrices and target matrix.

    Returns
    -------
    X_per_step : dict[h, np.ndarray]  shape (n_samples, n_lags + 7)
    Y          : np.ndarray           shape (n_samples, output_range)
    """
    n_features = n_lags + n_weather + 4   # lags + weather + time
    n          = len(values)
    n_samples  = n - n_lags - output_range + 1
    if n_samples <= 0:
        raise ValueError(f'Not enough data: need >{n_lags + output_range} points, got {n}')

    X_per_step = {h: np.empty((n_samples, n_features), dtype=np.float32)
                  for h in range(output_range)}
    Y = np.empty((n_samples, output_range), dtype=np.float32)

    for i in range(n_samples):
        lag_vals = values[i : i + n_lags].astype(np.float32)
        Y[i]     = values[i + n_lags : i + n_lags + output_range]

        for h in range(output_range):
            forecast_ts  = int(timestamps_ms[i + n_lags + h])
            weather_vec  = get_weather_vec(weather_lkp, forecast_ts, step_ms, n_weather)
            time_vec     = cyclic_time_features(forecast_ts)
            X_per_step[h][i] = np.concatenate([lag_vals, weather_vec, time_vec])

    return X_per_step, Y


# ── Prepare clean series ──────────────────────────────────────────────────────
values_clean  = interpolate_series(series_raw.values)
timestamps_ms = (series_raw.index.astype(np.int64) // 10**6).values
STEP_MS       = STEP_SECONDS * 1000

print('Building feature matrices (per-step)…')
X_per_step, Y_all = build_solar_dataset(
    values_clean, timestamps_ms, weather_lookup,
    N_LAGS, OUTPUT_RANGE, STEP_MS, N_WEATHER_FEATURES,
)

print(f'X_h shape (per step) : {X_per_step[0].shape}   (samples × {N_FEATURES} features)')
print(f'Y shape              : {Y_all.shape}   (samples × output_range)')
print(f'NaN in X_0           : {np.isnan(X_per_step[0]).sum()}')
print(f'NaN in Y             : {np.isnan(Y_all).sum()}')

# Quick sanity: feature breakdown
print(f'\nFeature breakdown:')
print(f'  Lags          : X[:, 0:{N_LAGS}]')
print(f'  shortwave_rad : X[:, {N_LAGS}]')
print(f'  temperature   : X[:, {N_LAGS+1}]')
print(f'  cloud_cover    : X[:, {N_LAGS+2}]')
print(f'  sin_hour      : X[:, {N_LAGS+3}]')
print(f'  cos_hour      : X[:, {N_LAGS+4}]')
print(f'  sin_doy       : X[:, {N_LAGS+5}]')
print(f'  cos_doy       : X[:, {N_LAGS+6}]')

## 5 · Train / Validation Split

In [ ]:
def walk_forward_splits(
    n_samples: int,
    n_splits: int,
    val_frac: float = 0.15,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Walk-forward splits — validation always follows training in time."""
    val_size  = max(1, int(n_samples * val_frac))
    min_train = n_samples - n_splits * val_size
    if min_train <= 0:
        raise ValueError('Too many splits for the available data; reduce N_CV_SPLITS.')
    splits = []
    for k in range(n_splits):
        end_train = min_train + k * val_size
        end_val   = end_train + val_size
        train_idx = np.arange(0, end_train)
        val_idx   = np.arange(end_train, min(end_val, n_samples))
        if len(val_idx) == 0:
            break
        splits.append((train_idx, val_idx))
    return splits


n_samples_total = len(X_per_step[0])
splits = walk_forward_splits(n_samples_total, N_CV_SPLITS)

train_idx_full, test_idx = splits[-1]
cv_splits = splits[:-1]

# X/Y for final train+test (step 0 used for shapes; real per-step split done in training)
Y_train = Y_all[train_idx_full]
Y_test  = Y_all[test_idx]

print(f'CV folds for tuning : {len(cv_splits)}')
print(f'Train samples       : {len(train_idx_full)}')
print(f'Test  samples       : {len(test_idx)}')

fig, ax = plt.subplots(figsize=(14, 2))
for fold_i, (tr, vl) in enumerate(cv_splits):
    ax.barh(fold_i, len(tr), color='steelblue', alpha=0.7, label='train' if fold_i == 0 else '')
    ax.barh(fold_i, len(vl), left=tr[-1], color='orange', alpha=0.7, label='val' if fold_i == 0 else '')
ax.barh(len(splits)-1, len(train_idx_full), color='steelblue', alpha=0.4)
ax.barh(len(splits)-1, len(test_idx), left=train_idx_full[-1], color='red', alpha=0.6, label='test')
ax.set_xlabel('Sample index')
ax.set_yticks(range(len(splits)))
ax.set_yticklabels([f'fold {i}' for i in range(len(splits)-1)] + ['final'])
ax.set_title('Walk-forward cross-validation splits')
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Hyperparameter Tuning

Randomised search on `TUNE_ON_STEP` (horizon h=0). Best params reused for all steps.

In [ ]:
PARAM_GRID = {
    'max_depth'        : [3, 4, 5, 6, 7],
    'eta'              : [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight' : [1, 3, 5, 10],
    'gamma'            : [0.0, 0.1, 0.3, 0.5],
    'lambda'           : [0.5, 1.0, 2.0, 5.0],
    'alpha'            : [0.0, 0.1, 0.5, 1.0],
}

FIXED_PARAMS = {
    'objective'  : 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'seed'       : 42,
    'nthread'    : -1,
    'verbosity'  : 0,
}
NUM_BOOST_ROUND = 300
EARLY_STOPPING  = 20


def cv_rmse_for_params(
    params: dict,
    X_steps: dict[int, np.ndarray],
    Y: np.ndarray,
    step_h: int,
    splits: list[tuple],
) -> float:
    """Average RMSE over walk-forward folds for one horizon step."""
    X = X_steps[step_h]
    y = Y[:, step_h]
    scores = []
    for tr_idx, vl_idx in splits:
        dtrain = xgb.DMatrix(X[tr_idx], label=y[tr_idx])
        dval   = xgb.DMatrix(X[vl_idx], label=y[vl_idx])
        bst    = xgb.train(
            {**FIXED_PARAMS, **params},
            dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            evals=[(dval, 'val')],
            early_stopping_rounds=EARLY_STOPPING,
            verbose_eval=False,
        )
        preds = bst.predict(dval)
        scores.append(float(np.sqrt(np.mean((preds - y[vl_idx]) ** 2))))
    return float(np.mean(scores))


rng     = np.random.RandomState(42)
sampler = list(ParameterSampler(PARAM_GRID, n_iter=N_TUNE_ITER, random_state=rng))

results = []
for i, candidate in enumerate(sampler):
    score = cv_rmse_for_params(candidate, X_per_step, Y_all, TUNE_ON_STEP, cv_splits)
    results.append({'params': candidate, 'cv_rmse': score})
    print(f'  [{i+1:2d}/{N_TUNE_ITER}] RMSE={score:.3f}  params={candidate}')

best       = min(results, key=lambda r: r['cv_rmse'])
BEST_PARAMS = {**FIXED_PARAMS, **best['params']}

print(f'\n✓ Best CV RMSE = {best["cv_rmse"]:.3f}')
print(f'  Best params  = {best["params"]}')

In [ ]:
scores_sorted = sorted(results, key=lambda r: r['cv_rmse'])
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(scores_sorted)), [r['cv_rmse'] for r in scores_sorted], width=0.8, color='steelblue')
ax.axhline(best['cv_rmse'], color='red', ls='--', lw=1.5, label=f'best = {best["cv_rmse"]:.3f}')
ax.set_xlabel('Candidate (sorted by RMSE)')
ax.set_ylabel('CV RMSE')
ax.set_title('Hyperparameter tuning — randomised search')
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Final Training — Multi-Step Direct Strategy

One XGBoost booster per horizon step `h ∈ {0, …, OUTPUT_RANGE−1}`.
Each booster is trained on its own feature matrix `X_per_step[h]`
(weather + time features at the target timestep).

In [ ]:
boosters: dict[int, xgb.Booster] = {}
best_rounds: list[int] = []
dtest_per_step: dict[int, xgb.DMatrix] = {}

for h in range(OUTPUT_RANGE):
    X_h = X_per_step[h]
    y_tr = Y_all[train_idx_full, h]
    y_te = Y_all[test_idx,       h]

    dtrain = xgb.DMatrix(X_h[train_idx_full], label=y_tr)
    dtest  = xgb.DMatrix(X_h[test_idx],       label=y_te)

    bst = xgb.train(
        BEST_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtest, 'test')],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False,
    )
    boosters[h] = bst
    best_rounds.append(bst.best_iteration)
    dtest_per_step[h] = dtest

    if h % 4 == 0 or h == OUTPUT_RANGE - 1:
        print(f'  step {h:2d}/{OUTPUT_RANGE-1}  best_round={bst.best_iteration}  '
              f'n_features={bst.num_features()}')

print(f'\n✓ All {OUTPUT_RANGE} boosters trained.')
print(f'  Avg best_round : {np.mean(best_rounds):.1f}')
print(f'  Features/booster: {boosters[0].num_features()}  (expected {N_FEATURES})')
assert boosters[0].num_features() == N_FEATURES, (
    f'Feature count mismatch! booster={boosters[0].num_features()} != N_FEATURES={N_FEATURES}'
)

## 8 · Evaluation

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1.0) -> float:
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return float('nan')
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


metrics_per_step = []
Y_pred = np.zeros_like(Y_test)

for h in range(OUTPUT_RANGE):
    preds  = boosters[h].predict(dtest_per_step[h])
    Y_pred[:, h] = preds
    y_true = Y_all[test_idx, h]
    metrics_per_step.append({
        'step'  : h,
        'MAE'   : float(np.mean(np.abs(preds - y_true))),
        'RMSE'  : float(np.sqrt(np.mean((preds - y_true) ** 2))),
        'MAPE_%': mape(y_true, preds),
    })

metrics_df = pd.DataFrame(metrics_per_step).set_index('step')
mean_mae   = metrics_df['MAE'].mean()
mean_rmse  = metrics_df['RMSE'].mean()
mean_mape  = metrics_df['MAPE_%'].mean()

print(f'Test-set metrics (mean over {OUTPUT_RANGE} steps)')
print(f'  MAE   = {mean_mae:.3f} MW')
print(f'  RMSE  = {mean_rmse:.3f} MW')
print(f'  MAPE  = {mean_mape:.2f}%')
print()
print(metrics_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, col in zip(axes, ['MAE', 'RMSE', 'MAPE_%']):
    ax.plot(metrics_df.index, metrics_df[col], marker='o', ms=4, color='steelblue')
    ax.set_xlabel('Horizon step')
    ax.set_ylabel(col)
    ax.set_title(f'{col} by horizon step')
plt.tight_layout()
plt.show()

# Actual vs predicted — a sample with real generation (find a daytime sample)
sample_idx = next(
    (i for i in range(len(Y_test)) if Y_test[i].mean() > 1.0),
    0
)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(Y_test[sample_idx],  label='Actual',    lw=1.5, color='goldenrod')
ax.plot(Y_pred[sample_idx],  label='Predicted', lw=1.5, ls='--', color='steelblue')
ax.set_title(f'Sample forecast vs actual (test sample #{sample_idx})')
ax.set_xlabel('Horizon step (hours)')
ax.set_ylabel('MW')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — averaged over all horizon boosters
importance_sum = np.zeros(N_FEATURES, dtype=float)
for bst in boosters.values():
    for feat_name, score in bst.get_score(importance_type='gain').items():
        idx = int(feat_name[1:])   # 'f0' → 0
        importance_sum[idx] += score
importance_avg = importance_sum / OUTPUT_RANGE

feat_names = (
    [f'lag_{N_LAGS - i}' for i in range(N_LAGS)]
    + ['GHI', 'temp', 'cloud_cover']
    + ['sin_hour', 'cos_hour', 'sin_doy', 'cos_doy']
)

top_k   = min(30, N_FEATURES)
top_idx = np.argsort(importance_avg)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['orange' if feat_names[i] in ('GHI','temp','cloud_cover')
          else ('green' if feat_names[i].startswith('sin') or feat_names[i].startswith('cos')
                else 'steelblue')
          for i in top_idx]
ax.bar(range(top_k), importance_avg[top_idx], color=colors)
ax.set_xticks(range(top_k))
ax.set_xticklabels([feat_names[i] for i in top_idx], rotation=45, ha='right', fontsize=8)
ax.set_title(f'Top-{top_k} feature importances (avg gain)  '
             f'[blue=lag, orange=weather, green=time]')
ax.set_ylabel('Avg gain')
plt.tight_layout()
plt.show()

## 9 · Bundle Assembly

Canonical layout expected by `ModelProvider` and `init_model`:
```
bundle/
  configuration/cache_config.json   ← model_type=solar + weather source
  model/
    xgb_model.json
    step_0.ubj  ...  step_23.ubj
```

In [ ]:
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

bundle_model_dir = BUNDLE_DIR / 'bundle' / 'model'
bundle_cfg_dir   = BUNDLE_DIR / 'bundle' / 'configuration'
bundle_model_dir.mkdir(parents=True)
bundle_cfg_dir.mkdir(parents=True)

# ── Save per-step boosters ────────────────────────────────────────────────────
step_files = []
for h, bst in sorted(boosters.items()):
    fname = f'step_{h}.ubj'
    bst.save_model(str(bundle_model_dir / fname))
    step_files.append(fname)

# ── Manifest ──────────────────────────────────────────────────────────────────
manifest = {'steps': step_files}
with open(bundle_model_dir / 'xgb_model.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ── cache_config.json — model_type=solar + weather source ────────────────────
cache_config = {
    'model_type'  : 'solar',
    'step'        : STEP_SECONDS,
    'input_range' : INPUT_RANGE,
    'output_range': OUTPUT_RANGE,
    'fallback'    : FALLBACK_MODEL,
    'sources': {
        'historical_data': {
            'type'   : 'historical_data',
            'url'    : SCADA_URL,
            'request': {'archive': ARCHIVES},
        },
        'weather': {
            'type'    : 'weather',
            'url'     : WEATHER_BASE_URL,
            'location': {'latitude': STATION_LAT, 'longitude': STATION_LON},
            'hours'   : OUTPUT_RANGE,
        },
    },
}

with open(bundle_cfg_dir / 'cache_config.json', 'w') as f:
    json.dump(cache_config, f, indent=2)

print('Bundle contents:')
total_mb = 0.0
for p in sorted(BUNDLE_DIR.rglob('*')):
    if p.is_file():
        sz = p.stat().st_size / 1024
        total_mb += sz / 1024
        print(f'  {p.relative_to(BUNDLE_DIR)}  ({sz:.1f} KB)')
print(f'\nTotal: {total_mb:.2f} MB')

## 10 · MLflow Logging & Model Registration

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run() as run:
    run_id = run.info.run_id

    mlflow.log_params({
        'model_type'     : 'solar',
        'n_lags'         : N_LAGS,
        'n_weather_feats': N_WEATHER_FEATURES,
        'n_time_feats'   : N_TIME_FEATURES,
        'n_features'     : N_FEATURES,
        'output_range'   : OUTPUT_RANGE,
        'input_range'    : INPUT_RANGE,
        'step_seconds'   : STEP_SECONDS,
        'train_days'     : TRAIN_HISTORY_DAYS,
        'archives'       : json.dumps(ARCHIVES),
        'station_lat'    : STATION_LAT,
        'station_lon'    : STATION_LON,
        'weather_vars'   : json.dumps(WEATHER_VARIABLES),
        'fallback'       : FALLBACK_MODEL,
        'num_boost_round': NUM_BOOST_ROUND,
        'best_round_avg' : round(float(np.mean(best_rounds)), 1),
        **{f'hp_{k}': v for k, v in best['params'].items()},
    })

    mlflow.log_metrics({
        'test_mae_mean' : round(mean_mae,  4),
        'test_rmse_mean': round(mean_rmse, 4),
        'test_mape_mean': round(mean_mape, 4),
        'cv_rmse_best'  : round(best['cv_rmse'], 4),
    })

    for h, row in metrics_df.iterrows():
        mlflow.log_metrics({
            f'test_mae_h{h:02d}' : round(row['MAE'],    4),
            f'test_rmse_h{h:02d}': round(row['RMSE'],   4),
            f'test_mape_h{h:02d}': round(row['MAPE_%'], 4),
        })

    # Log bundle directory as artifact
    bundle_src = BUNDLE_DIR / 'bundle'
    mlflow.log_artifacts(str(bundle_src), artifact_path='bundle')

    # Log metrics CSV
    metrics_csv = BUNDLE_DIR / 'metrics_per_step.csv'
    metrics_df.reset_index().to_csv(metrics_csv, index=False)
    mlflow.log_artifact(str(metrics_csv))

    mlflow.set_tags({
        'model_type'     : 'solar',
        'target_archive' : ARCHIVES[0],
        'station_lat'    : STATION_LAT,
        'station_lon'    : STATION_LON,
        'n_features'     : N_FEATURES,
        'weather_source' : WEATHER_BASE_URL,
    })

    print(f'Run logged  → run_id = {run_id}')
    print(f'Experiment  → {EXPERIMENT_NAME}')

In [ ]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

try:
    client.create_registered_model(REGISTERED_MODEL)
    print(f'Created registered model "{REGISTERED_MODEL}"')
except mlflow.exceptions.MlflowException:
    print(f'Registered model "{REGISTERED_MODEL}" already exists')

model_version = client.create_model_version(
    name=REGISTERED_MODEL,
    source=f'runs:/{run_id}/bundle',
    run_id=run_id,
    description=(
        f'Solar XGBoost multi-step direct | n_lags={N_LAGS} | n_feat={N_FEATURES} | '
        f'output={OUTPUT_RANGE}h | lat={STATION_LAT} lon={STATION_LON} | '
        f'MAE={mean_mae:.2f} RMSE={mean_rmse:.2f} MAPE={mean_mape:.1f}%'
    ),
)
version_number = model_version.version
print(f'Model version {version_number} registered.')

client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias=REGISTER_ALIAS,
    version=version_number,
)
print(f'Alias "{REGISTER_ALIAS}" → version {version_number}')

print()
print('═══ Ready for inference ═══')
print(f'  model_id      = {REGISTERED_MODEL}')
print(f'  version_alias = {REGISTER_ALIAS}')
print(f'  GET /predict/{REGISTERED_MODEL}?version_alias={REGISTER_ALIAS}')

## 11 · Smoke-test via SolarAdapter

Loads the bundle locally and runs a forward pass through `SolarAdapter`
with a synthetic weather payload — exactly as the inference worker would.

In [ ]:
import sys

SRC_PATH = Path('..') / 'src'
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from adapters.config import load_model_config
from adapters.adapters import SolarAdapter
from adapters.base_interface import PredictionInput

bundle_path = (BUNDLE_DIR / 'bundle').resolve()
cfg = load_model_config('__smoke_test__', bundle_path=bundle_path)
print(f'Config loaded: model_type={cfg.model_type}  step={cfg.step}s  '
      f'input={cfg.input_range}  output={cfg.output_range}')
assert cfg.model_type == 'solar', f'Expected model_type=solar, got {cfg.model_type}'
assert cfg.weather_lat == STATION_LAT
assert cfg.weather_lon == STATION_LON

# Load boosters and wrap in SolarAdapter
manifest_path = bundle_path / 'model' / 'xgb_model.json'
with open(manifest_path) as f:
    manifest_data = json.load(f)

models_dict = {}
for step_fname in manifest_data['steps']:
    bst = xgb.Booster()
    bst.load_model(str(bundle_path / 'model' / step_fname))
    step_idx = int(step_fname.split('_')[-1].split('.')[0])
    models_dict[step_idx] = bst

adapter = SolarAdapter(model_name='smoke_test', legacy_model=models_dict)
adapter.load()
print(f'\nSolarAdapter loaded: {len(models_dict)} boosters, '
      f'{adapter.legacy_model[0].num_features()} features/booster')

# Build a realistic inference input
sample_lags  = values_clean[-N_LAGS:].tolist()
sample_ts_ms = int(timestamps_ms[-N_LAGS])

# Synthetic weather payload matching inference format
smoke_weather = {
    'hourly': [
        {
            'solar_radiation': max(0.0, 400.0 * math.sin(math.pi * h / 12) if 6 <= h <= 18 else 0.0),
            'temperature'     : 20.0 + 5.0 * math.sin(math.pi * h / 12),
            'cloud_cover'         : 20.0,
        }
        for h in range(OUTPUT_RANGE)
    ]
}

pred_input = PredictionInput(
    features=sample_lags,
    metadata={
        'step'        : STEP_MS,
        'output_range': OUTPUT_RANGE,
        'timestamps'  : [sample_ts_ms],
        'weather_data': smoke_weather,
    },
)

output = adapter.predict(pred_input)

print(f'\nSmoke-test predictions ({len(output.predictions)} steps):')
print(f'  min  = {min(output.predictions):.3f} MW')
print(f'  max  = {max(output.predictions):.3f} MW')
print(f'  mean = {float(np.mean(output.predictions)):.3f} MW')
assert all(p >= 0.0 for p in output.predictions), 'Negative predictions found!'
assert output.metadata['model_type'] == 'solar'
assert output.metadata['weather_used'] is True
print('\n✓ Smoke-test passed — bundle is ready for deployment.')